In [1]:
import pandas as pd
import numpy as np

## **Zinn+ 2022**

### Data Description

Table 6:

```
   Bytes Format Units   Label        Explanations
   
   1- 12 A12    ---     ID           Unique identifier (1)
  14- 22 I9     ---     EPIC         [201051317/251632350] EPIC identifier
  24- 25 I2     ---     C            [1/18] K2 Campaign
  27- 31 I5     ---     Prior        [1/21899]? K2 GAP target priority for
                                      a given K2 campaign (2)
  33- 39 A7     ---     EV           Evolutionary state (3)
  41- 47 A7     ---     EV-A2Z       A2Z pipeline evolutionary state (3)
  49- 55 A7     ---     EV-CAN       CAN pipeline evolutionary state (3)
  57- 63 A7     ---     EV-COR       COR pipeline evolutionary state (3)
  65- 71 A7     ---     EV-SYD       SYD pipeline evolutionary state (3)
  73- 79 A7     ---     EV-BAM       BAM pipeline evolutionary state (3)
  81- 87 A7     ---     EV-BHM       BHM pipeline evolutionary state (3)
  89- 95 F7.3   uHz     numax-A2Z-o  [3.29/300]? Raw A2Z numax
  97-103 F7.3   uHz     numax-BAM-o  [3.5/250]? Raw BAM numax
 105-111 F7.3   uHz     numax-BHM-o  [5/300]? Raw BHM numax
 113-119 F7.3   uHz     numax-CAN-o  [2.9/299]? Raw CAN numax
 121-127 F7.3   uHz     numax-COR-o  [1.8/244]? Raw COR numax
 129-135 F7.3   uHz     numax-SYD-o  [3/283]? Raw SYD numax
 137-142 F6.3   uHz   e_numax-A2Z-o  [0.15/62]? Uncertainty in numax-A2Z-o
 144-150 F7.3   uHz   e_numax-BAM-o  [0.05/113]? Uncertainty in numax-BAM-o
 152-157 F6.3   uHz   e_numax-BHM-o  [0.048/55]? Uncertainty in numax-BHM-o
 159-164 F6.3   uHz   e_numax-CAN-o  [0.1/31]? Uncertainty in numax-CAN-o
 166-170 F5.3   uHz   e_numax-COR-o  [0.1/6]? Uncertainty in numax-COR-o
 172-178 F7.3   uHz   e_numax-SYD-o  [0/400]? Uncertainty in numax-SYD-o
 180-185 F6.3   uHz     dnu-A2Z-o    [0.56/22.5]? Raw A2Z dnu
 187-192 F6.3   uHz     dnu-BAM-o    [0.004/32]? Raw BAM dnu
 194-199 F6.3   uHz     dnu-BHM-o    [0.77/26]? Raw BHM dnu
 201-206 F6.3   uHz     dnu-CAN-o    [0.87/24]? Raw CAN dnu
 208-213 F6.3   uHz     dnu-COR-o    [0.4/20]? Raw COR dnu
 215-220 F6.3   uHz     dnu-SYD-o    [0.7/19.2]? Raw SYD dnu
 222-227 F6.3   uHz   e_dnu-A2Z-o    [0.001/12]? Uncertainty in dnu-A2Z-o
 229-233 F5.3   uHz   e_dnu-BAM-o    [0/7]? Uncertainty in dnu-BAM-o
 235-239 F5.3   uHz   e_dnu-BHM-o    [0.003/3]? Uncertainty in dnu-BHM-o
 241-245 F5.3   uHz   e_dnu-CAN-o    [0.004/1.1]? Uncertainty in dnu-CAN-o
 247-251 F5.3   uHz   e_dnu-COR-o    [0.02/0.4]? Uncertainty in dnu-COR-o
 253-258 F6.3   uHz   e_dnu-SYD-o    [0.01/27.4]? Uncertainty in dnu-SYD-o
```

### CDS X-Matching

Read-in xmatch.

In [2]:
array_full = pd.read_csv("../0_inputdata_CDSxmatch/Zinn2022_cdsxmatch.csv")

/tmp/ipykernel_14536/1780656891.py:1: DtypeWarning: Columns (178: ClassELS) have mixed types. Specify dtype option on import or set low_memory=False.
  array_full = pd.read_csv("../0_inputdata_CDSxmatch/Zinn2022_cdsxmatch.csv")


Convert EV state, with:

 - 0 ==> not specified
 - 1 ==> likely RGB
 - 2 ==> likely CHeB

In [3]:
array_full["EV"] = array_full["EV"].replace({
    "nan": int(0),
    "RGB": int(1),
    "RGB/AGB": int(0),
    "RC": int(2)
})

Average pipelines for $\Delta\nu$ and $\nu_\mathrm{max}$ values.

In [4]:
def mean_std_func(values, errors):
    vals = np.stack([array_full[c] for c in values])
    errs = np.stack([array_full[c] for c in errors])

    errs[(errs <= 0) | ~np.isfinite(errs)] = np.nan # set invalid uncertainties to NaN

    vals = np.where(np.isfinite(errs), vals, np.nan)

    mean = np.nanmean(vals, axis=0)
    std = np.nanstd(errs, axis=0)

    return mean, std

In [5]:
numax_cols = ["numax-A2Z-o", "numax-BAM-o", "numax-BHM-o", "numax-CAN-o", "numax-COR-o", "numax-SYD-o"]
enumax_cols = ["e_numax-A2Z-o", "e_numax-BAM-o", "e_numax-BHM-o", "e_numax-CAN-o", "e_numax-COR-o", "e_numax-SYD-o"]

dnu_cols = ["dnu-A2Z-o", "dnu-BAM-o", "dnu-BHM-o", "dnu-CAN-o", "dnu-COR-o", "dnu-SYD-o"]
ednu_cols = ["e_dnu-A2Z-o", "e_dnu-BAM-o", "e_dnu-BHM-o", "e_dnu-CAN-o", "e_dnu-COR-o", "e_dnu-SYD-o"]

numax, e_numax = mean_std_func(numax_cols, enumax_cols)
dnu, e_dnu = mean_std_func(dnu_cols, ednu_cols)

/tmp/ipykernel_14536/131846934.py:9: RuntimeWarning: Mean of empty slice
  mean = np.nanmean(vals, axis=0)
/home/iaindc/.pyenv/versions/3.11.17/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


Curate simplfied array.

In [6]:
z22 = pd.DataFrame({
    # star identifiers
    "gaiadr3_id": array_full["Source"],
    "ra": array_full["RAdeg"],
    "dec": array_full["DEdeg"],
    # asteroseismic parameters
    "numax": numax,
    "e_numax": e_numax,
    "dnu": dnu,
    "e_dnu": e_dnu,
    "ev_state": array_full["EV"],
    # Gaia parameters
    "GAIA-teff": array_full["Teff"],
    "GAIA-e_teff": (array_full["B_Teff"] - array_full["b_Teff"]) / 2,
    "GAIA-logg": array_full["logg"],
    "GAIA-e_logg": (array_full["B_logg"] - array_full["b_logg"]) / 2,
    "GAIA-fe_h": array_full["[Fe/H]"],
    "GAIA-e_fe_h": (array_full["B_[Fe/H]"] - array_full["b_[Fe/H]"]) / 2,
    # GSP Spec parameters
    "GSPSPEC-teff": array_full["Teff-S"],
    "GSPSPEC-e_teff": (array_full["B_Teff-S"] - array_full["b_Teff-S"]) / 2,
    "GSPSPEC-logg": array_full["logg-S"],
    "GSPSPEC-e_logg": (array_full["B_logg-S"] - array_full["b_logg-S"]) / 2,
    "GSPSPEC-fe_h": array_full["[Fe/H]-S"],
    "GSPSPEC-e_fe_h": (array_full["B_[Fe/H]-S"] - array_full["b_[Fe/H]-S"]) / 2,
    "GSPSPEC-si_fe": array_full["[Si/Fe]-S"],
    "GSPSPEC-e_si_fe": (array_full["B_[Si/Fe]-S"] - array_full["b_[Si/Fe]-S"]) / 2,
    "GSPSPEC-ca_fe": array_full["[Ca/Fe]-S"],
    "GSPSPEC-e_ca_fe": (array_full["B_[Ca/Fe]-S"] - array_full["b_[Ca/Fe]-S"]) / 2,
    "GSPSPEC-ti_fe": array_full["[Ti/Fe]-S"],
    "GSPSPEC-e_ti_fe": (array_full["B_[Ti/Fe]-S"] - array_full["b_[Ti/Fe]-S"]) / 2,
    "GSPSPEC-mg_fe": array_full["[Mg/Fe]-S"],
    "GSPSPEC-e_mg_fe": (array_full["B_[Mg/Fe]-S"] - array_full["b_[Mg/Fe]-S"]) / 2,
    "GSPSPEC-nd_fe": array_full["[Nd/Fe]-S"],
    "GSPSPEC-e_nd_fe": (array_full["B_[Nd/Fe]-S"] - array_full["b_[Nd/Fe]-S"]) / 2,
    "GSPSPEC-feii_m": array_full["[FeII/M]-S"],
    "GSPSPEC-e_feii_m": (array_full["B_[FeII/M]-S"] - array_full["b_[FeII/M]-S"]) / 2,
    "GSPSPEC-s_fe": array_full["[S/Fe]-S"],
    "GSPSPEC-e_s_fe": (array_full["B_[S/Fe]-S"] - array_full["b_[S/Fe]-S"]) / 2,
    "GSPSPEC-zr_fe": array_full["[Zr/Fe]-S"],
    "GSPSPEC-e_zr_fe": (array_full["B_[Zr/Fe]-S"] - array_full["b_[Zr/Fe]-S"]) / 2,
    "GSPSPEC-n_fe": array_full["[N/Fe]-S"],
    "GSPSPEC-e_n_fe": (array_full["B_[N/Fe]-S"] - array_full["b_[N/Fe]-S"]) / 2,
    "GSPSPEC-cr_fe": array_full["[Cr/Fe]-S"],
    "GSPSPEC-e_cr_fe": (array_full["B_[Cr/Fe]-S"] - array_full["b_[Cr/Fe]-S"]) / 2,
    "GSPSPEC-ce_fe": array_full["[Ce/Fe]-S"],
    "GSPSPEC-e_ce_fe": (array_full["B_[Ce/Fe]-S"] - array_full["b_[Ce/Fe]-S"]) / 2,
    "GSPSPEC-ni_fe": array_full["[Ni/Fe]-S"],
    "GSPSPEC-e_ni_fe": (array_full["B_[Ni/Fe]-S"] - array_full["b_[Ni/Fe]-S"]) / 2
})

z22 = z22.dropna(subset=["e_numax", "e_dnu"])

Taken within $0.75\times$ and $1.35\times$ of $\Delta\nu$-$\nu_\mathrm{max}$ relation (Stello+ 09, Mosser+ 11):

$$ \Delta\nu = 0.27 \nu_\mathrm{max}^{0.76}. $$

In [7]:
def relation(numax):
    coeff = 0.27
    return coeff * numax**(0.76)

mask = (z22["dnu"] < 1.35*relation(z22["numax"])) & (z22["dnu"] > 0.75*relation(z22["numax"]))

z22 = z22[mask]

Write to `.csv` file.

In [8]:
z22.to_csv("./zinn22_asteroseismic.csv", index=False)